# 08. Agent Harness

## Learning Objectives
- Understand the concept and role of AgentHarness
- Learn the harness's core capabilities: planning, filesystem access, and task delegation
- Understand context management through offloading and summarization
- Configure code execution and Human-in-the-Loop
- Connect skills and memory systems


In [ ]:
# Environment setup
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set!"
print("Environment setup complete")


In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")

print(f"Model configured: {model.model_name}")


---
## 1. AgentHarness Concept

**AgentHarness** is a **comprehensive capability provider** for long-running autonomous agents.  
It bundles together the infrastructure needed for complex multi-step agent work.

### Core Capabilities Provided by the Harness

| Capability | Description |
|------|------|
| **Planning** | Manage structured task lists with `write_todos` |
| **Filesystem** | Read, write, and search files in virtual or local environments |
| **Task Delegation** | Delegate work through subagents |
| **Context Management** | Compress context through offloading and summarization |
| **Code Execution** | Run code safely in sandboxed environments |
| **Human-in-the-Loop** | Require approval for sensitive operations |
| **Skills & Memory** | Use specialized workflows and persistent knowledge |

When you call `create_deep_agent()`, all of these pieces are assembled into a single agent.


In [ ]:
# AgentHarness concept — create_deep_agent assembles the harness
harness_config = {
    "model": "gpt-4.1",
    "system_prompt": "You are a project management assistant.",
    "planning": True,
    "filesystem": True,
    "subagents": [],
    "context_management": True,
}

print("AgentHarness components:")
for key, value in harness_config.items():
    print(f"  {key}: {value}")


---
## 2. Planning Tools

The agent uses the `write_todos` tool to break complex work into a **structured task list**.  
Each task has a status:

| Status | Description |
|------|------|
| `pending` | Not started yet |
| `in_progress` | Currently in progress |
| `completed` | Finished |


In [ ]:
# write_todos example — structured task list
todo_list = [
    {"task": "Analyze the project structure", "status": "completed"},
    {"task": "Design the API endpoints", "status": "in_progress"},
    {"task": "Write the database schema", "status": "pending"},
    {"task": "Write the tests", "status": "pending"},
    {"task": "Document the project", "status": "pending"},
]

print("=== Agent task list ===")
for i, item in enumerate(todo_list, 1):
    icon = {"completed": "[x]", "in_progress": "[-]", "pending": "[ ]"}
    print(f"  {icon[item['status']]} {i}. {item['task']}")


---
## 3. Virtual Filesystem

The harness supports standard file operations through configurable filesystem backends.

| Tool | Description |
|------|------|
| `ls` | List directory contents with metadata |
| `read_file` | Read file contents with line numbers (and image support) |
| `write_file` | Create files |
| `edit_file` | Replace strings inside files |
| `glob` | Search for files by pattern |
| `grep` | Search file contents in different output modes |
| `execute` | Run shell commands (sandbox backends only) |


In [ ]:
# Example filesystem tool calls (reference only)
fs_operations = {
    "ls": 'ls(path="/project/src")',
    "read_file": 'read_file(path="/project/src/main.py")',
    "write_file": 'write_file(path="/project/config.yaml", content="debug: true")',
    "edit_file": 'edit_file(path="/project/src/main.py", old="v1", new="v2")',
    "glob": 'glob(pattern="**/*.py")',
    "grep": 'grep(pattern="TODO", path="/project/src")',
}

print("=== Filesystem tool call examples ===")
for tool_name, call_example in fs_operations.items():
    print(f"  {tool_name:12s} -> {call_example}")


---
## 4. Task Delegation — Subagents

The harness allows the main agent to create **temporary subagents** for isolated multi-step work.

### Advantages of Subagents

| Advantage | Description |
|------|------|
| **Context isolation** | Subagent execution does not pollute the main context |
| **Parallel execution** | Multiple subagents can run at the same time |
| **Specialization** | Each subagent can get its own tools and prompt |
| **Token efficiency** | The main agent receives a compressed result |


In [ ]:
# Example subagent delegation configuration (reference only)
subagent_config = [
    {
        "name": "researcher",
        "description": "Investigates information using web search.",
        "system_prompt": "Summarize search results concisely.",
        "tools": ["internet_search"],
    },
    {
        "name": "coder",
        "description": "Writes and tests code.",
        "system_prompt": "Write clean and testable code.",
        "tools": ["write_file", "execute"],
    },
]

print("=== Subagent configuration ===")
for sa in subagent_config:
    print(f"  [{sa['name']}] {sa['description']}")
    print(f"    tools: {', '.join(sa['tools'])}")


---
## 5. Context Management

The biggest challenge for long-running agents is the **context window limit**.  
The harness addresses it with two main techniques.

### Input Context Assembly
The initial prompt is assembled from the system prompt, instructions, memory guidelines, skill information, and filesystem documentation.

### Runtime Context Compression

| Technique | Behavior | Trigger |
|------|------|--------|
| **Offloading** | Stores content larger than 20,000 tokens on disk and keeps only pointers in context | Based on content size |
| **Summarization** | Compresses conversation history into a structured summary | Triggered when the model window limit is approached |

The original data is preserved in filesystem storage, so information is not lost.


In [ ]:
# Example context-management settings (reference only)
context_config = {
    "offloading": {
        "enabled": True,
        "threshold_tokens": 20000,
        "storage": "filesystem",
    },
    "summarization": {
        "enabled": True,
        "trigger": "window_limit_approach",
        "preserve_original": True,
    },
}

print("=== Context management settings ===")
for section, settings in context_config.items():
    print(f"\n[{section}]")
    for key, value in settings.items():
        print(f"  {key}: {value}")


---
## 6. Code Execution

Sandbox backends expose the `execute` tool, which runs commands in an isolated environment.  
That improves safety, cleanliness, and reproducibility without affecting the host system.


In [ ]:
# Example sandboxed execute calls (reference only)
execute_examples = [
    {"command": "python -c 'print(2+2)'", "desc": "Run a Python snippet"},
    {"command": "pip install requests", "desc": "Install a package"},
    {"command": "pytest tests/", "desc": "Run the test suite"},
]

print("=== Sandbox execute tool examples ===")
for ex in execute_examples:
    print(f"  $ {ex['command']}")
    print(f"    -> {ex['desc']}")


---
## 7. Human-in-the-Loop

You can require human approval for selected tool calls through interrupt settings.


In [ ]:
# Example Human-in-the-Loop configuration (reference only)
hitl_config = {
    "interrupt_on": {
        "write_file": True,
        "edit_file": True,
        "execute": True,
    }
}

print("=== Human-in-the-Loop configuration ===")
print("Tools that require approval:")
for tool, enabled in hitl_config["interrupt_on"].items():
    status = "approval required" if enabled else "automatic"
    print(f"  {tool}: {status}")

print("\nDecision options: approve, reject, edit")


---
## 8. Skills and Memory

### Skills
Skills are specialized workflows that follow the **Agent Skills standard**.  
They are loaded progressively when relevant, which reduces token usage.

- Each skill is defined in a `SKILL.md` file
- Skills are activated when the triggering conditions match
- They package tools, prompts, and workflows together

### Memory
Memory uses **`AGENTS.md`**-style persistent context files.  
It stores reusable guidelines, preferences, and project knowledge beyond a single conversation.

| Scope | Location | Range |
|------|------|------|
| Global memory | `~/.deepagents/<agent>/memories/` | All projects |
| Project memory | `.deepagents/AGENTS.md` | Current project |


In [ ]:
# Example skill and memory configuration (reference only)
skills_config = [
    {"name": "code-review", "trigger": "when the user asks for a code review"},
    {"name": "test-writer", "trigger": "when the user asks for tests"},
    {"name": "doc-generator", "trigger": "when the user asks for documentation"},
]

memory_config = {
    "global": "~/.deepagents/my-agent/memories/",
    "project": ".deepagents/AGENTS.md",
}

print("=== Skill configuration ===")
for skill in skills_config:
    print(f"  [{skill['name']}] trigger: {skill['trigger']}")

print("\n=== Memory configuration ===")
for scope, path in memory_config.items():
    print(f"  {scope}: {path}")


---
## Summary

| Topic | Core Concept | Key API / Tool |
|------|----------|-------------|
| Harness concept | Comprehensive capability provider for long-running agents | `create_deep_agent()` |
| Planning tools | Structured task list management | `write_todos` |
| Filesystem | Virtual and local file operations | `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, `execute` |
| Subagents | Isolated task delegation, parallel execution | `subagents`, `task` |
| Context management | Offloading (20K tokens), summarization | automatic |
| Code execution | Safe command execution in sandboxes | `execute` |
| HITL | Human approval for sensitive tool calls | `interrupt_on` |
| Skills / Memory | Specialized workflows + persistent context | `SKILL.md`, `AGENTS.md` |

### Next Steps
→ Continue to **[09_comparison.ipynb](./09_comparison.ipynb)**


---
**References:**
- [Deep Agents Harness](../docs/deepagents/05-harness.md)
